# G17 AMFU-Net — Focal Tversky & Boundary-Aware loss variations (Kaggle)

Trains **U-Net + DAG** (the 6.19M-parameter best architecture) with new loss variations and tests every run on the **exact same 266-scan test set** as Exp-16–82 (pinned to the image list saved with Exp-16; those patients are excluded from training).

**Before running**
1. *Settings → Accelerator → GPU* (T4 or P100). *Settings → Internet → On* (the dataset and old checkpoints come from Hugging Face; both are public, no token needed).
2. Choose runs in `RUN_ONLY` (or leave `None` for all), then **Save Version → Save & Run All (Commit)** so it keeps running with the browser closed (12 h limit per session).
3. Results land in `/kaggle/working/runs/<run>/` and `/kaggle/working/results_summary.csv` (download from the version's *Output* tab).

**Loss definitions**
- `dice`: BCE + soft Dice (Exp-16 family)
- `ft`: BCE + Focal Tversky, TI = (TP+1)/(TP+αFP+βFN+1), loss = (1−TI)^(1/γ) (Exp-32–47 family)
- `+ λ·boundary`: soft Dice between morphological boundary maps (dilation − erosion) of prediction and mask, with optional delay/ramp warm-up (Ahorts' `boundary_focal_tversky`, `wu10-20`, `bndXXX` branches)

> The boundary term is rebuilt from the `extract_boundary` helper in `training_marimo.py` and the `boundary_weight` / `boundary_warmup_*` keys in the Hugging Face configs. Ahorts' exact training code for it is not on GitHub. Ask him to confirm it matches before comparing these runs to Exp-48–82.

In [ ]:
# 1) Setup
!pip -q install datasets safetensors huggingface_hub
import torch; print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU – turn on the accelerator")

In [ ]:
# ============================================================
# Core code: data, model, losses, training, evaluation
# (architecture + split copied from training/training_marimo.py)
# ============================================================
import gc, json, math, os, random, time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from scipy.ndimage import binary_erosion, distance_transform_edt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG = 256
MODALITY_COL = {"raw": "raw_image", "clahe": "clahe_image", "srad": "srad_image", "enhanced": "image"}


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


# ---------------- Data: the exact 266-scan test set used by Exp-16–82 ----------------
# StratifiedGroupKFold(shuffle=True) gives different folds on different scikit-learn versions
# (on Kaggle it yields 255 test scans, not 266). So the test set is pinned to the image list
# saved with Exp-16, every patient in it is excluded from train/val, and the rest is split as before.
def load_splits(data_repo="Ahorts/G17-preprocessed-dataset", seed=42,
                model_repo="Ahorts/G17-AMFU-Net-Paper",
                pin_branch="exp16-unet-dag-gpu-reflect-raw-cosine-only"):
    from datasets import concatenate_datasets, load_dataset
    from huggingface_hub import hf_hub_download
    from sklearn.model_selection import StratifiedGroupKFold
    hf = load_dataset(data_repo, revision="11b552497ffa661fa16c58c95bd7415b31073721")  # dataset as used for Exp-16–82
    allds = concatenate_datasets([hf["train"], hf["validation"], hf["test"]])
    ref = pd.read_csv(hf_hub_download(model_repo, "results/internal_test_per_image.csv", revision=pin_branch))
    want = set(zip(ref["dataset"].astype(str), ref["image_id"].astype(str)))
    keys = list(zip(np.asarray(allds["dataset"]).astype(str), np.asarray(allds["image_id"]).astype(str)))
    test_idx = np.array([i for i, k in enumerate(keys) if k in want])
    assert len(test_idx) == len(want) == 266, f"pinned test set mismatch: found {len(test_idx)} of {len(want)}"
    groups = np.asarray(allds["patient_id"]).astype(str)
    test_patients = set(groups[test_idx])
    rem = np.array([i for i in range(len(allds)) if groups[i] not in test_patients])
    labels = np.array([f"{d}_{c}" for d, c in zip(allds["dataset"], allds["class_label"])])
    tr, va = next(StratifiedGroupKFold(9, shuffle=True, random_state=seed + 1).split(rem, labels[rem], groups[rem]))
    split = {"train": allds.select(rem[tr].tolist()), "validation": allds.select(rem[va].tolist()),
             "test": allds.select(test_idx.tolist())}
    assert not (set(groups[rem[tr]]) | set(groups[rem[va]])) & test_patients, "patient leakage"
    print({k: len(v) for k, v in split.items()}, "| test pinned to", pin_branch)
    print(pd.Series(split["test"]["dataset"]).value_counts().to_dict())
    return split


_TENSOR_CACHE = {}


def to_tensors(ds, modality, split_name):
    """Decode + resize once per (split, modality), keep uint8 on GPU like the original pipeline."""
    key = (split_name, modality)
    if key in _TENSOR_CACHE:
        return _TENSOR_CACHE[key]
    col = MODALITY_COL[modality]
    imgs, msks = [], []
    for r in ds:
        imgs.append(cv2.resize(np.asarray(r[col].convert("L")), (IMG, IMG), interpolation=cv2.INTER_AREA))
        msks.append(cv2.resize(np.asarray(r["mask"].convert("L")), (IMG, IMG), interpolation=cv2.INTER_NEAREST))
    out = {"img": torch.from_numpy(np.stack(imgs))[:, None].to(DEVICE),
           "msk": torch.from_numpy(np.stack(msks))[:, None].to(DEVICE),
           "dataset": list(ds["dataset"]), "image_id": list(ds["image_id"])}
    _TENSOR_CACHE[key] = out
    return out


def norm(img_u8):
    return (img_u8.float() / 255.0 - 0.5) / 0.5


def augment(img, msk):
    """Same augmentation as training: hflip, rot ±15, scale 0.9–1.1, translate ±10%,
    96-px reflection pad for the image, zero fill for the mask."""
    if random.random() < 0.5:
        img, msk = TF.hflip(img), TF.hflip(msk)
    angle, scale = random.uniform(-15, 15), random.uniform(0.9, 1.1)
    tx, ty = random.uniform(-0.1, 0.1) * IMG, random.uniform(-0.1, 0.1) * IMG
    p = 96
    imgp = F.pad(img[None].float(), (p, p, p, p), mode="reflect")[0]
    imgp = TF.affine(imgp, angle=angle, translate=[tx, ty], scale=scale, shear=0.0,
                     interpolation=TF.InterpolationMode.BILINEAR)
    img = imgp[:, p:p + IMG, p:p + IMG]
    msk = TF.affine(msk, angle=angle, translate=[tx, ty], scale=scale, shear=0.0,
                    interpolation=TF.InterpolationMode.NEAREST)
    return (img / 255.0 - 0.5) / 0.5, (msk > 127).float()


# ---------------- Model ----------------
class ConvBNReLU(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.block = nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.block(x)


class DAG(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.branch_average = nn.Conv2d(c, c, 3, padding=1)
        self.branch_maximum = nn.Conv2d(c, c, 3, padding=1)
        self.output = nn.Conv2d(c, c, 3, padding=1)

    def forward(self, x):
        g1 = F.adaptive_avg_pool2d(torch.sigmoid(self.branch_average(x)), 1)
        g2 = F.adaptive_max_pool2d(torch.sigmoid(self.branch_maximum(x)), 1)
        return self.output(x * g1 + x * g2)


class MFEF(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.conv1 = nn.Conv2d(c, c, 1); self.conv3 = nn.Conv2d(c, c, 3, padding=1)
        self.conv5 = nn.Conv2d(c, c, 5, padding=2); self.gap_projection = nn.Conv2d(c, c, 1)
        self.compress = nn.Conv2d(c * 4, c, 1); self.dilated = nn.Conv2d(c, c, 3, padding=2, dilation=2)

    def forward(self, x):
        g = F.interpolate(self.gap_projection(F.adaptive_avg_pool2d(x, 1)), size=x.shape[-2:],
                          mode="bilinear", align_corners=False)
        w1 = torch.softmax(F.relu(self.compress(torch.cat([self.conv1(x), self.conv3(x), self.conv5(x), g], 1))), 1)
        w2 = torch.softmax(F.relu(self.dilated(x)), 1)
        return x * (0.5 * w1 + 0.5 * w2)


class AMFUNet(nn.Module):
    def __init__(self, base_channels=32, use_dag=True, use_mfef=False, **_):
        super().__init__()
        b = base_channels; self.use_dag, self.use_mfef = use_dag, use_mfef
        self.enc1, self.enc2 = ConvBNReLU(1, b), ConvBNReLU(b, b * 2)
        self.enc3, self.enc4 = ConvBNReLU(b * 2, b * 4), ConvBNReLU(b * 4, b * 8)
        self.bottleneck = ConvBNReLU(b * 8, b * 16); self.pool = nn.MaxPool2d(2)
        if use_mfef:
            self.mfef = MFEF(b * 16)
        if use_dag:
            self.dag1, self.dag2, self.dag3, self.dag4 = DAG(b), DAG(b * 2), DAG(b * 4), DAG(b * 8)
        self.up4, self.dec4 = nn.ConvTranspose2d(b * 16, b * 8, 2, 2), ConvBNReLU(b * 16, b * 8)
        self.up3, self.dec3 = nn.ConvTranspose2d(b * 8, b * 4, 2, 2), ConvBNReLU(b * 8, b * 4)
        self.up2, self.dec2 = nn.ConvTranspose2d(b * 4, b * 2, 2, 2), ConvBNReLU(b * 4, b * 2)
        self.up1, self.dec1 = nn.ConvTranspose2d(b * 2, b, 2, 2), ConvBNReLU(b * 2, b)
        self.head = nn.Conv2d(b, 1, 1)

    def forward(self, x):
        s1 = self.enc1(x); s2 = self.enc2(self.pool(s1)); s3 = self.enc3(self.pool(s2)); s4 = self.enc4(self.pool(s3))
        x = self.bottleneck(self.pool(s4))
        if self.use_mfef:
            x = self.mfef(x)
        for up, dec, s, k in ((self.up4, self.dec4, s4, 4), (self.up3, self.dec3, s3, 3),
                              (self.up2, self.dec2, s2, 2), (self.up1, self.dec1, s1, 1)):
            x = dec(torch.cat([up(x), getattr(self, f"dag{k}")(s) if self.use_dag else s], 1))
        return self.head(x)


# ---------------- Losses ----------------
def extract_boundary(t):
    """Differentiable morphological gradient (dilation − erosion), as in training_marimo.py."""
    return F.max_pool2d(t, 3, 1, 1) - (-F.max_pool2d(-t, 3, 1, 1))


def boundary_weight_at(epoch, lam, delay=0, ramp=0):
    """λ schedule: 0 until `delay` epochs, then linear ramp to λ over `ramp` epochs (Ahorts' wu10-20 runs)."""
    if lam <= 0:
        return 0.0
    if epoch < delay:
        return 0.0
    if ramp <= 0:
        return lam
    return lam * min(1.0, (epoch - delay) / ramp)


def compute_loss(logits, labels, cfg, bnd_w):
    logits, labels = logits.float(), labels.float()
    p = torch.sigmoid(logits)
    dims = (1, 2, 3)
    loss = F.binary_cross_entropy_with_logits(logits, labels)
    if cfg["base"] == "dice":                      # BCE + soft Dice (Exp-16 family)
        inter = (p * labels).sum(dims)
        dice = (2 * inter + 1.0) / (p.sum(dims) + labels.sum(dims) + 1.0)
        loss = loss + (1.0 - dice.mean())
    elif cfg["base"] == "ft":                      # BCE + Focal Tversky (Exp-32–47 family)
        a, b, g = cfg["alpha"], cfg["beta"], cfg["gamma"]
        tp = (p * labels).sum(dims); fp = (p * (1 - labels)).sum(dims); fn = ((1 - p) * labels).sum(dims)
        ti = (tp + 1.0) / (tp + a * fp + b * fn + 1.0)
        loss = loss + torch.pow(torch.clamp(1 - ti, 1e-7, 1.0), 1.0 / g).mean()
    if bnd_w > 0:                                   # boundary-aware term: soft Dice between boundary maps
        pb, lb = extract_boundary(p), extract_boundary(labels)
        bd = (2 * (pb * lb).sum(dims) + 1.0) / (pb.sum(dims) + lb.sum(dims) + 1.0)
        loss = loss + bnd_w * (1.0 - bd.mean())
    return loss


# ---------------- Metrics (identical definitions to training) ----------------
def surface_distances(pred, tgt):
    pred, tgt = pred.astype(bool), tgt.astype(bool)
    if not pred.any() and not tgt.any():
        return 0.0, 0.0
    if not pred.any() or not tgt.any():
        d = float(math.hypot(*tgt.shape)); return d, d
    pb = pred ^ binary_erosion(pred, border_value=0); tb = tgt ^ binary_erosion(tgt, border_value=0)
    pb = pb if pb.any() else pred; tb = tb if tb.any() else tgt
    d = np.concatenate([distance_transform_edt(~tb)[pb], distance_transform_edt(~pb)[tb]])
    return float(np.percentile(d, 95)), float(d.mean())


@torch.no_grad()
def predict(model, data, bs=32):
    model.eval(); out = []
    for i in range(0, len(data["img"]), bs):
        with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
            out.append(torch.sigmoid(model(norm(data["img"][i:i + bs])).float()).cpu())
    return torch.cat(out)[:, 0].numpy()


def fast_dice(probs, data, thr=0.5):
    pred = probs >= thr; gt = data["msk"][:, 0].cpu().numpy() > 127
    tp = (pred & gt).sum((1, 2)); fp = (pred & ~gt).sum((1, 2)); fn = (~pred & gt).sum((1, 2))
    return float(np.mean((2 * tp + 1e-7) / (2 * tp + fp + fn + 1e-7)))


def full_metrics(probs, data, thr=0.5):
    gt_all = data["msk"][:, 0].cpu().numpy() > 127; rows = []
    for i, (pr, gt) in enumerate(zip(probs >= thr, gt_all)):
        tp = (pr & gt).sum(); fp = (pr & ~gt).sum(); fn = (~pr & gt).sum()
        hd95, asd = surface_distances(pr, gt)
        rows.append({"image_id": data["image_id"][i], "dataset": data["dataset"][i],
                     "dice": (2 * tp + 1e-7) / (2 * tp + fp + fn + 1e-7),
                     "jaccard": (tp + 1e-7) / (tp + fp + fn + 1e-7),
                     "recall": (tp + 1e-7) / (tp + fn + 1e-7), "hd95": hd95, "asd": asd})
    return pd.DataFrame(rows)


def summarise(df, name):
    rows = []
    for grp, d in [("Overall", df)] + list(df.groupby("dataset")):
        rows.append({"run": name, "cohort": grp, "n": len(d), "dice": d.dice.mean() * 100,
                     "iou": d.jaccard.mean() * 100, "hd95": d.hd95.mean(), "asd": d.asd.mean(),
                     "dice_sd": d.dice.std() * 100})
    return pd.DataFrame(rows)


# ---------------- Training ----------------
def train_run(cfg, split, out_root, epochs, batch=4, accum=3, lr=1e-3, wd=1e-5, min_lr=1e-5,
              eval_every=250, use_amp=True):
    name = cfg["name"]; out = Path(out_root) / name; out.mkdir(parents=True, exist_ok=True)
    if (out / "test_per_image.csv").exists():
        print(f"[skip] {name} already finished"); return pd.read_csv(out / "test_per_image.csv")
    seed_everything(cfg.get("seed", 42))
    tr = to_tensors(split["train"], cfg["modality"], "train")
    va = to_tensors(split["validation"], cfg["modality"], "validation")
    te = to_tensors(split["test"], cfg["modality"], "test")
    model = AMFUNet(use_dag=cfg.get("dag", True), use_mfef=cfg.get("mfef", False)).to(DEVICE)
    nparams = sum(p.numel() for p in model.parameters())
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    n = len(tr["img"]); micro_per_epoch = math.ceil(n / batch)
    steps_per_epoch = math.ceil(micro_per_epoch / accum); total = steps_per_epoch * epochs
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total, eta_min=min_lr)
    scaler = torch.amp.GradScaler(enabled=use_amp and DEVICE.type == "cuda")
    best, best_step, step, log = -1.0, 0, 0, []
    print(f"\n=== {name} | {cfg['modality']} | base={cfg['base']} λ={cfg.get('bnd', 0)} | "
          f"params={nparams:,} | {epochs} ep × {steps_per_epoch} steps = {total} steps ===")
    t0 = time.time()
    for ep in range(epochs):
        model.train(); perm = torch.randperm(n).tolist()
        bw = boundary_weight_at(ep, cfg.get("bnd", 0.0), cfg.get("bnd_delay", 0), cfg.get("bnd_ramp", 0))
        opt.zero_grad(set_to_none=True)
        for mi in range(micro_per_epoch):
            ids = perm[mi * batch:(mi + 1) * batch]
            pairs = [augment(tr["img"][j], tr["msk"][j]) for j in ids]
            x = torch.stack([a for a, _ in pairs]); y = torch.stack([b for _, b in pairs])
            with torch.autocast(DEVICE.type, enabled=use_amp and DEVICE.type == "cuda"):
                logits = model(x)
            loss = compute_loss(logits, y, cfg, bw) / accum
            scaler.scale(loss).backward()
            if (mi + 1) % accum == 0 or mi + 1 == micro_per_epoch:
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step(); step += 1
                if step % eval_every == 0 or step == total:
                    vd = fast_dice(predict(model, va), va); model.train()
                    log.append({"step": step, "epoch": ep + 1, "val_dice": vd, "lr": sched.get_last_lr()[0], "bnd_w": bw})
                    if vd > best:
                        best, best_step = vd, step
                        torch.save(model.state_dict(), out / "best.pt")
        if ep == 0 or (ep + 1) % 10 == 0:
            el = time.time() - t0
            print(f"  ep {ep+1:>3}/{epochs}  val_dice(best)={best:.4f}@{best_step}  "
                  f"{el/60:.1f} min elapsed, ETA {el/(ep+1)*(epochs-ep-1)/60:.0f} min")
    model.load_state_dict(torch.load(out / "best.pt", map_location=DEVICE))
    df = full_metrics(predict(model, te), te)
    df.to_csv(out / "test_per_image.csv", index=False)
    pd.DataFrame(log).to_csv(out / "val_log.csv", index=False)
    json.dump({**cfg, "params": nparams, "best_val_dice": best, "best_step": best_step, "epochs": epochs,
               "minutes": (time.time() - t0) / 60}, open(out / "config.json", "w"), indent=2)
    print(summarise(df, name).round(2).to_string(index=False))
    del model, opt; gc.collect(); torch.cuda.empty_cache()
    return df


# ---------------- Re-evaluate existing Hugging Face checkpoints on the same split ----------------
def eval_hf_checkpoint(branch, split, repo="Ahorts/G17-AMFU-Net-Paper"):
    from huggingface_hub import hf_hub_download
    from safetensors.torch import load_file
    c = json.load(open(hf_hub_download(repo, "config.json", revision=branch)))
    model = AMFUNet(use_dag=c.get("use_dag", True), use_mfef=c.get("use_mfef", False)).to(DEVICE)
    model.load_state_dict(load_file(hf_hub_download(repo, "model.safetensors", revision=branch)), strict=True)
    modality = ("enhanced" if "enhanced" in branch else "clahe" if "clahe" in branch
                else "srad" if "srad" in branch else "raw")
    te = to_tensors(split["test"], modality, "test")
    return full_metrics(predict(model, te), te)


In [ ]:
# 2) Experiment plan — edit freely
SMOKE_TEST = False         # set True only for a quick 2-epoch pipeline check
EPOCHS = 2 if SMOKE_TEST else 200
OUT = "/kaggle/working/runs"

FT = dict(base="ft", alpha=0.3, beta=0.7, gamma=1.33)   # the team's current Focal Tversky setting
PLAN = [
    # --- Reproducibility control (Exp-16 rerun): tells you how much run-to-run noise there is ---
    dict(name="K01_dice_raw_repro",            modality="raw", base="dice"),
    # --- Focal Tversky variations on raw (Exp-39 used α0.3/β0.7/γ1.33 → 84.70%) ---
    dict(name="K02_ft_a05_b05_g133_raw",       modality="raw", base="ft", alpha=0.5, beta=0.5, gamma=1.33),  # symmetric, focal only
    dict(name="K03_ft_a06_b04_g133_raw",       modality="raw", base="ft", alpha=0.6, beta=0.4, gamma=1.33),  # FP-heavy: tests the 'expands into shadow on raw' explanation
    dict(name="K04_ft_a04_b06_g133_raw",       modality="raw", base="ft", alpha=0.4, beta=0.6, gamma=1.33),  # milder FN penalty
    dict(name="K05_tversky_a03_b07_g1_raw",    modality="raw", base="ft", alpha=0.3, beta=0.7, gamma=1.0),   # plain Tversky (no focal)
    dict(name="K06_ft_a03_b07_g2_raw",         modality="raw", base="ft", alpha=0.3, beta=0.7, gamma=2.0),   # stronger focusing
    # --- Boundary-aware variations on raw (Exp-49 λ0.5 → 84.96%, Exp-80 λ0.25 → 85.16%) ---
    dict(name="K07_dice_bnd025_raw",           modality="raw", base="dice", bnd=0.25),                      # boundary on top of the Exp-16 loss (new)
    dict(name="K08_dice_bnd05_raw",            modality="raw", base="dice", bnd=0.5),                       # (new)
    dict(name="K09_ft_bnd10_raw",              modality="raw", **FT, bnd=1.0),                              # λ=1.0 on raw (only tried on enhanced, Exp-73)
    dict(name="K10_ft_a05_b05_bnd025_wu_raw",  modality="raw", base="ft", alpha=0.5, beta=0.5, gamma=1.33,
         bnd=0.25, bnd_delay=10, bnd_ramp=20),                                                                  # symmetric FT + warmed-up boundary
    # --- CLAHE, where Focal Tversky helped most (Exp-38 → 84.94%) ---
    dict(name="K11_ft_a04_b06_g133_clahe",     modality="clahe", base="ft", alpha=0.4, beta=0.6, gamma=1.33),
    dict(name="K12_dice_bnd025_clahe",         modality="clahe", base="dice", bnd=0.25),
]
RUN_ONLY = ["K01_dice_raw_repro", "K03_ft_a06_b04_g133_raw", "K07_dice_bnd025_raw", "K08_dice_bnd05_raw"]  # None = all runs
todo = [c for c in PLAN if RUN_ONLY is None or c["name"] in RUN_ONLY]
print(f"{len(todo)} runs × {EPOCHS} epochs")

In [ ]:
# 3) Load data (same split as all previous experiments) — downloads ~1 GB once
split = load_splits()

In [ ]:
# 4) Train + test every planned run (finished runs are skipped, so you can re-run safely)
results = []
for cfg in todo:
    df = train_run(cfg, split, OUT if not SMOKE_TEST else OUT + "_smoke", epochs=EPOCHS)
    results.append(summarise(df, cfg["name"]))
summary = pd.concat(results)
summary.to_csv("/kaggle/working/results_summary.csv" if not SMOKE_TEST else "/kaggle/working/smoke_summary.csv", index=False)
summary[summary.cohort == "Overall"].sort_values("dice", ascending=False).round(2)

In [ ]:
# 5) Re-score the team's existing checkpoints on the identical test split (apples-to-apples reference rows)
REFERENCE = {
    "Exp-16 dice raw":         "exp16-unet-dag-gpu-reflect-raw-cosine-only",
    "Exp-25 dice enhanced (app)": "exp25-unet-dag-gpu-reflect-enhanced-cosine-only",
    "Exp-49 ft+bnd0.5 raw":    "exp33-unet-dag-boundary-raw-cosine-only",   # HF branch name uses the old numbering
    "Exp-67 ft+bnd0.5 wu raw": "exp67-unet-dag-boundary-raw-wu10-20",
    "Exp-80 ft+bnd0.25 raw":   "exp80-unet-dag-boundary-raw-bnd025",
}
ref = pd.concat([summarise(eval_hf_checkpoint(b, split), n) for n, b in REFERENCE.items()])
ref.to_csv("/kaggle/working/reference_summary.csv", index=False)
ref[ref.cohort == "Overall"].round(2)

In [ ]:
# 6) One leaderboard: new runs + reference checkpoints, overall and per cohort
board = pd.concat([ref.assign(kind="reference"), summary.assign(kind="new")])
board.to_csv("/kaggle/working/leaderboard.csv", index=False)
overall = board[board.cohort == "Overall"].sort_values("dice", ascending=False)
display(overall[["run", "kind", "dice", "dice_sd", "iou", "hd95", "asd"]].round(2))
display(board.pivot_table(index="run", columns="cohort", values=["dice", "hd95"]).round(2))
print("Note: single seed per run. Differences under ~0.5–1% Dice are likely noise — compare against K01 vs Exp-16.")